# Molecular Dynamics of a Pt–OH–H₂O Interface with SpookyNet


## Goal of the notebook

Let us follow a short molecular dynamics trajectory of a Pt–OH–H₂O interface. A pretrained SpookyNet potential supplies the forces, and ASE advances the atoms through time.


## Description of the method

We will hold the Pt slab fixed, assign thermal velocities to the mobile water, and use a Bussi thermostat for an NVT trajectory. The example uses a low temperature so we can inspect local motion.

Neighbor lists, distances, angles, species labels, and speeds give us several views of the trajectory. We will check each geometric definition before we use it to discuss chemistry.


## What we will do

1. Let us inspect the interface, identify fixed atoms, and label water molecules.
2. We will attach SpookyNet and run NVT dynamics with recorded settings and random seeds.
3. We will analyze distances, angles, species assignments, bond correlations, and speeds.


In [ ]:
# Install course dependencies only in Google Colab. Local runs use environment.yml.
!if python -c "import google.colab" >/dev/null 2>&1; then pip install -q ase torch matplotlib numpy git+https://github.com/OUnke/SpookyNet.git; fi

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import torch

from ase.constraints import FixAtoms
from ase.io import read, Trajectory
from ase.md import MDLogger
from ase.md.bussi import Bussi
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.neighborlist import NeighborList
from ase.units import fs
from ase.visualize import view
from IPython.display import display
from pathlib import Path
from spookynet.spookynet_calculator import SpookyNetCalculator

In [ ]:
# Standard publication figure sizes in inches
plt.rc("figure", figsize  = (8.25 / 2.54, 8.25 / 2.54))
plt.rc("figure", dpi      = 100)
plt.rc("lines",  linewidth= 4)
plt.rc("font",   size     = 10)
plt.rc("font",   family   = "sans-serif")
plt.rc("axes",   titlesize= 10)
plt.rc("axes",   labelsize= 9)
plt.rc("xtick",  labelsize= 8)
plt.rc("ytick",  labelsize= 8)
plt.rc("legend", fontsize = 8)
plt.rc("figure", titlesize= 10)

pd.options.display.float_format = "{:.2f}".format

In [ ]:
method = "spookynet"
prefix = "Pt-OH-H2O"

## Physical context

Let us use molecular dynamics to explore an interface as a sequence of atomic configurations. For a literature example, see [molecular-dynamics study](https://doi.org/10.1016/j.cattod.2012.06.001).


## Load and inspect the model

The `.traj` file stores a Pt(111) slab with a mixed OH–H₂O overlayer. The index `-1` selects the final frame, which contains the most relaxed geometry. The `view()` command opens an interactive structure display.


In [ ]:
# Download once into the current folder.
!if [ ! -s Pt-OH-H2O.traj ]; then wget -q https://github.com/doublelayer/test_models/raw/refs/heads/main/Pt-w/Pt-OH-H2O.traj -O Pt-OH-H2O.traj; fi
if not Path('Pt-OH-H2O.traj').is_file() or Path('Pt-OH-H2O.traj').stat().st_size == 0:
    raise FileNotFoundError("Download failed: Pt-OH-H2O.traj")

Let us inspect the model: a mixed OH–H₂O overlayer sits on a Pt(111) surface.


In [ ]:
atoms = read(f"{prefix}.traj", -1)
view(atoms, viewer="x3d")

## Prepare constraints and species labels

The ASE surface tags identify the Pt layers. We keep every Pt slab atom fixed so the water motion is easy to interpret.


In [ ]:
# Hold every tagged Pt slab layer fixed during molecular dynamics.
atoms.set_constraint(FixAtoms(mask=atoms.get_tags() > 0))
atoms.set_pbc(True)

This cell identifies H₂O molecules by locating each O atom with exactly two H neighbors within 1.0 Å. It then assigns `tag=4` to that O atom and both H atoms. Later, we use this label to separate H₂O from OH during analysis.


In [ ]:
# enumerate gives both the item number (starting at zero) and the item itself.
for i, at_i in enumerate(atoms):
    if at_i.symbol != "O":
        continue
    h_neighbors = []
    for j, at_j in enumerate(atoms):
        if at_j.symbol == "H" and atoms.get_distance(i, j) < 1.0:
            h_neighbors.append(j)
    # Two short O–H contacts identify an intact water molecule.
    if len(h_neighbors) == 2:
        atoms[i].tag = 4  # Custom water label, assigned using the O–H neighbor test.
        for j in h_neighbors:
            atoms[j].tag = 4  # Custom water label, assigned using the O–H neighbor test.

## Configure SpookyNet

We load the pretrained SpookyNet neural network from a `.pth` file and attach it to the structure. ASE will request energies and forces from this network as it moves the atoms. The settings `charge=0` and `magmom=0` specify the electronic state.


In [ ]:
# The installed package and its trained weights are separate files.
model_path = Path("spookynet_parameters.pth")
!if [ ! -s spookynet_parameters.pth ]; then wget -q https://raw.githubusercontent.com/OUnke/SpookyNet/d57b1fc02c4f1304a9445b2b9aa55a906818dd1b/parameters.pth -O spookynet_parameters.pth; fi
if not model_path.is_file() or model_path.stat().st_size == 0:
    raise FileNotFoundError("SpookyNet weights are missing; rerun the download cell.")
calculator = SpookyNetCalculator(
    load_from=str(model_path), charge=0, magmom=0
)
atoms.calc = calculator

## Run NVT molecular dynamics

Let us set up three parts of the trajectory:

- `MaxwellBoltzmannDistribution` assigns initial velocities for the target temperature.
- The Bussi thermostat samples the NVT ensemble through stochastic velocity rescaling.
- A 1 fs time step advances the atoms by one femtosecond per step. With 1,000 steps, the trajectory spans 1 ps.


> **Pause and predict:** What temperature or structural changes would alert you to an unstable trajectory?
>
> **What to look for:** Check the energy, temperature, and atomic geometry before you interpret a correlation.


The command `random.seed` sets the starting state of the pseudorandom number generator. This choice makes the initial random values reproducible across runs.


In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
temperature_k = 10
number_of_md_steps = 1000
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_k, force_temp=True)
dynamics = Bussi(atoms, timestep=1.0 * fs, temperature_K=temperature_k, taut=0.5 * fs)
trajectory = Trajectory(f"{prefix}_MD.traj", "w", atoms)
logger = MDLogger(dynamics, atoms, f"{prefix}_MD.log", mode="w", header=True, stress=False, peratom=True)
dynamics.attach(logger, interval=10)
dynamics.attach(trajectory.write, interval=10)
dynamics.run(number_of_md_steps)

In [ ]:
trajectory = Trajectory(f"{prefix}_MD.traj")
print("Number of frames:", len(trajectory))
print("Number of atoms:", len(trajectory[0]))
print(trajectory[0])

### Read the trajectory length

Let us distinguish integration steps from saved frames. Saving every tenth step reduces the number of stored frames; it leaves the simulated time unchanged.


In [ ]:
print("Saved frames:", len(trajectory))
print("Requested duration / fs:", number_of_md_steps)
print("This duration uses the configured 1 fs step; it is not the animation playback time.")

## Analyze and validate the trajectory

This cell builds a neighbor list for the final frame. Two atoms count as neighbors when their separation falls within the sum of their 3 Å radii, for a pair cutoff of 6 Å.

> **Before you begin:** State the physical question or failure mode that each analysis checks.
>
> **What to look for:** Check the neighbor definitions and species labels against the chemistry before you interpret a correlation.


In [ ]:
final_structure = trajectory[-1]
number_of_atoms = len(final_structure)
cutoffs = [3.0] * number_of_atoms
neighbor_list = NeighborList(cutoffs, self_interaction=False, bothways=True)

Inspect the final MD frame before you calculate summary statistics. Look for desorption, dissociation, overlapping atoms, or slab distortion.


In [ ]:
view(final_structure, viewer="x3d")

For every atom in every frame, this cell finds the nearest neighbor with a 3.0 Å cutoff and plots the distance distribution. Use the histogram to check whether bond lengths look plausible and whether the structure remains intact.


In [ ]:
nearest_neighbor_distances_angstrom = []
for atoms in trajectory:
    neighbor_list.update(atoms)
    cell = atoms.get_cell()
    positions = atoms.positions
    for i in range(number_of_atoms):
        indices, offsets = neighbor_list.get_neighbors(i)
        if len(indices) == 0:
            continue
        displacements_angstrom = positions[indices] + offsets @ cell - positions[i]
        distances_angstrom = np.linalg.norm(displacements_angstrom, axis=1)
        nearest_neighbor_distances_angstrom.append(distances_angstrom.min())
nearest_neighbor_distances_angstrom = np.array(nearest_neighbor_distances_angstrom)
plt.hist(nearest_neighbor_distances_angstrom, bins=100)
plt.xlabel("nearest-neighbour distance / Å")
plt.ylabel("count")
plt.title("Histogram of nearest-neighbour distances")

# Close the canvas here; display the completed figure on the next slide.
figure_for_slide = plt.gcf()
plt.close(figure_for_slide)

In [ ]:
display(figure_for_slide)

For each O atom with a Pt neighbor within 3.0 Å and an H neighbor within 1.2 Å, this cell calculates the Pt–O–H angle and O–H bond length. The plot tests whether OH orientation tracks its bond length, as a possible sign of surface trans-influence. Blue points show all O atoms; red points show O atoms labeled as H₂O.


In [ ]:
pt_o_h_angles_deg = []
oh_bond_lengths_angstrom = []
tagged_pt_o_h_angles_deg = []
tagged_oh_bond_lengths_angstrom = []

In [ ]:
for atoms in trajectory:
    neighbor_list.update(atoms)
    cell = atoms.get_cell()
    positions_angstrom = atoms.positions
    symbols = atoms.get_chemical_symbols()
    tags = atoms.get_tags()
    for i in range(number_of_atoms):
        if symbols[i] != "O":
            continue
        indices, offsets = neighbor_list.get_neighbors(i)
        if len(indices) == 0:
            continue
        displacements_angstrom = (positions_angstrom[indices] + offsets @ cell - positions_angstrom[i])
        distances_angstrom = np.linalg.norm(displacements_angstrom, axis=1)
        pt_indices = []
        for j, idx in enumerate(indices):
            if symbols[idx] == "Pt" and distances_angstrom[j] < 3.0:
                pt_indices.append(j)
        h_indices = []
        for j, idx in enumerate(indices):
            if symbols[idx] == "H" and distances_angstrom[j] < 1.2:
                h_indices.append(j)
        for jp in pt_indices:
            for jh in h_indices:
                v_pt = displacements_angstrom[jp]
                v_h = displacements_angstrom[jh]
                d_oh = np.linalg.norm(v_h)
                if d_oh == 0:
                    continue
                cos_theta = np.dot(v_pt, v_h) / (np.linalg.norm(v_pt) * np.linalg.norm(v_h))
                cos_theta = np.clip(cos_theta, -1.0, 1.0)
                theta_deg = np.degrees(np.arccos(cos_theta))
                pt_o_h_angles_deg.append(theta_deg)
                oh_bond_lengths_angstrom.append(d_oh)
                if tags[i] == 4:
                    tagged_pt_o_h_angles_deg.append(theta_deg)
                    tagged_oh_bond_lengths_angstrom.append(d_oh)

In [ ]:
pt_o_h_angles_deg = np.array(pt_o_h_angles_deg)
oh_bond_lengths_angstrom = np.array(oh_bond_lengths_angstrom)
tagged_pt_o_h_angles_deg = np.array(tagged_pt_o_h_angles_deg)
tagged_oh_bond_lengths_angstrom = np.array(tagged_oh_bond_lengths_angstrom)
correlation_all = np.corrcoef(pt_o_h_angles_deg, oh_bond_lengths_angstrom)[0, 1]
print("Correlation(angle, O-H length) for all O =", correlation_all)
if len(tagged_pt_o_h_angles_deg) > 1:
    correlation_tagged = np.corrcoef(tagged_pt_o_h_angles_deg, tagged_oh_bond_lengths_angstrom)[0, 1]
    print("Correlation(angle, O-H length) for O (tag=4) =", correlation_tagged)
else:
    print("No data for O atoms with tag=4.")
plt.scatter(pt_o_h_angles_deg, oh_bond_lengths_angstrom, s=5, alpha=0.3)
plt.scatter(tagged_pt_o_h_angles_deg, tagged_oh_bond_lengths_angstrom, s=5, alpha=0.3)
plt.xlabel("Pt-O-H angle / deg")
plt.ylabel("O-H bond length / Å")
plt.title("Correlation between Pt-O-H angle and O-H bond length")

# Close the canvas here; display the completed figure on the next slide.
figure_for_slide = plt.gcf()
plt.close(figure_for_slide)

In [ ]:
display(figure_for_slide)

This geometric classifier uses two features:

- OH has one bonded H and often a small spread in the surface-normal direction.
- H₂O has two bonded H atoms and often extends farther from the surface plane.

Plot the number of H neighbors against the vertical spread to estimate the OH and H₂O populations across the trajectory.


In [ ]:
hydrogen_neighbors_per_oxygen = []
plane_spread_angstrom = []

In [ ]:
# zip pairs items from the supplied lists in the same order.
for atoms in trajectory:
    neighbor_list.update(atoms)
    cell = atoms.get_cell()
    positions_angstrom = atoms.positions
    symbols = atoms.get_chemical_symbols()
    for i in range(number_of_atoms):
        if symbols[i] != "O":
            continue
        neighbor_indices, offsets = neighbor_list.get_neighbors(i)
        if len(neighbor_indices) == 0:
            continue
        displacements_angstrom = (positions_angstrom[neighbor_indices] + offsets @ cell - positions_angstrom[i])
        distances_angstrom = np.linalg.norm(displacements_angstrom, axis=1)
        neighbor_symbols = []
        for j in neighbor_indices:
            neighbor_symbols.append(symbols[j])
        h_idx = []
        for j, (s, d) in enumerate(zip(neighbor_symbols, distances_angstrom)):
            if s == "H" and d < 1.2:
                h_idx.append(j)
        pt_idx = []
        for j, (s, d) in enumerate(zip(neighbor_symbols, distances_angstrom)):
            if s == "Pt" and d < 2.8:
                pt_idx.append(j)
        n_h = len(h_idx)
        hydrogen_neighbors_per_oxygen.append(n_h)
        group_coords = [positions_angstrom[i, 1]]
        for hydrogen_neighbor in h_idx:
            group_coords.append(positions_angstrom[neighbor_indices[hydrogen_neighbor], 1])
        if len(pt_idx) > 0:
            j = pt_idx[np.argmin(distances_angstrom[pt_idx])]
            group_coords.append(positions_angstrom[neighbor_indices[j], 1])
        if len(group_coords) > 1:
            spread_y = np.ptp(group_coords)
        else:
            spread_y = 0.0
        plane_spread_angstrom.append(spread_y)

In [ ]:
hydrogen_neighbors_per_oxygen = np.array(hydrogen_neighbors_per_oxygen)
plane_spread_angstrom = np.array(plane_spread_angstrom)
plt.scatter(hydrogen_neighbors_per_oxygen, plane_spread_angstrom, s=8, alpha=0.3)
plt.xlabel("number of H neighbours per O")
plt.ylabel("y-coordinate spread (Å)")
plt.title("Structural classification of O centres")

# Close the canvas here; display the completed figure on the next slide.
figure_for_slide = plt.gcf()
plt.close(figure_for_slide)

In [ ]:
display(figure_for_slide)

In [ ]:
is_oh = (hydrogen_neighbors_per_oxygen == 1) & (plane_spread_angstrom < 0.5)
is_h2o = hydrogen_neighbors_per_oxygen == 2
print("Approximate OH atoms:", np.sum(is_oh))
print("Approximate H2O atoms:", np.sum(is_h2o))
plt.hist(plane_spread_angstrom[is_oh], bins=30, alpha=0.6, label="OH (planar)")
plt.hist(plane_spread_angstrom[is_h2o], bins=30, alpha=0.6, label="H2O (non-planar)")
plt.xlabel("y-spread (Å)")
plt.ylabel("count")
plt.legend()

# Close the canvas here; display the completed figure on the next slide.
figure_for_slide = plt.gcf()
plt.close(figure_for_slide)

In [ ]:
display(figure_for_slide)

This cell tests the proposed trans-influence through Pt–O and O–H bond lengths. The fitted correlation is weak (r ≈ −0.14 for all O atoms). Its sign agrees with the proposed effect; its size alone gives little support. We can test the idea with longer sampling, uncertainty estimates, and a sensitivity check for the geometric filter. The `pt_mask` uses a 1.9 Å cutoff for directly bonded Pt–O pairs.


In [ ]:
pt_o_lengths_angstrom = []
o_h_lengths_angstrom = []
tagged_pt_o_lengths_angstrom = []
tagged_o_h_lengths_angstrom = []

In [ ]:
for atoms in trajectory:
    neighbor_list.update(atoms)
    cell = atoms.get_cell()
    positions_angstrom = atoms.positions
    symbols = atoms.get_chemical_symbols()
    tags = atoms.get_tags()
    for i in range(number_of_atoms):
        if symbols[i] != "O":
            continue
        neighbor_indices, offsets = neighbor_list.get_neighbors(i)
        if len(neighbor_indices) == 0:
            continue
        displacements_angstrom = (positions_angstrom[neighbor_indices] + offsets @ cell - positions_angstrom[i])
        distances_angstrom = np.linalg.norm(displacements_angstrom, axis=1)
        neighbor_symbols = []
        for j in neighbor_indices:
            neighbor_symbols.append(symbols[j])
        h_mask_values = []
        pt_mask_values = []
        for symbol, distance_angstrom in zip(neighbor_symbols, distances_angstrom):
            h_mask_values.append(symbol == "H" and distance_angstrom < 1.1)
            pt_mask_values.append(symbol == "Pt" and distance_angstrom < 1.9)
        h_mask = np.array(h_mask_values)
        pt_mask = np.array(pt_mask_values)
        h_indices = np.where(h_mask)[0]
        pt_indices = np.where(pt_mask)[0]
        if len(h_indices) == 1 and len(pt_indices) >= 1:
            jh = h_indices[0]
            d_oh = distances_angstrom[jh]
            for jp in pt_indices:
                d_po = distances_angstrom[jp]
                pt_o_lengths_angstrom.append(d_po)
                o_h_lengths_angstrom.append(d_oh)
                if tags[i] == 4:
                    tagged_pt_o_lengths_angstrom.append(d_po)
                    tagged_o_h_lengths_angstrom.append(d_oh)

In [ ]:
pt_o_lengths_angstrom = np.array(pt_o_lengths_angstrom)
o_h_lengths_angstrom = np.array(o_h_lengths_angstrom)
tagged_pt_o_lengths_angstrom = np.array(tagged_pt_o_lengths_angstrom)
tagged_o_h_lengths_angstrom = np.array(tagged_o_h_lengths_angstrom)
print("Number of Pt-OH triplets (all O):", len(pt_o_lengths_angstrom))
correlation_all = np.corrcoef(pt_o_lengths_angstrom, o_h_lengths_angstrom)[0, 1]
print("Correlation(Pt-O length, O-H length) for all O =", correlation_all)
if len(tagged_pt_o_lengths_angstrom) > 1:
    print("Number of Pt-OH triplets (O tag=4):", len(tagged_pt_o_lengths_angstrom))
    correlation_tagged = np.corrcoef(tagged_pt_o_lengths_angstrom, tagged_o_h_lengths_angstrom)[0, 1]
    print("Correlation(Pt-O length, O-H length) for O (tag=4) =", correlation_tagged)
else:
    print("No data for O atoms with tag=4.")
plt.scatter(pt_o_lengths_angstrom, o_h_lengths_angstrom, s=5, alpha=0.3)
plt.scatter(tagged_pt_o_lengths_angstrom, tagged_o_h_lengths_angstrom, s=5, alpha=0.3)
plt.xlabel("Pt-O bond length (Å)")
plt.ylabel("O-H bond length (Å)")
plt.title("Pt-OH: correlation between Pt-O and O-H bond lengths")

# Close the canvas here; display the completed figure on the next slide.
figure_for_slide = plt.gcf()
plt.close(figure_for_slide)

In [ ]:
display(figure_for_slide)

This cell gathers the speed of each water atom across all frames and plots its distribution. At thermal equilibrium, the speeds should follow a Maxwell–Boltzmann distribution at the target temperature. At 10 K, expect a narrow distribution. Check the sampled kinetic temperature and distribution shape as well.


In [ ]:
water_speed_magnitudes = []
for atoms in trajectory:
    velocities = atoms.get_velocities()
    if velocities is None:
        continue
    for i, atom in enumerate(atoms):
        if atom.tag == 4:
            water_speed_magnitudes.append(np.linalg.norm(velocities[i]))
water_speed_magnitudes = np.array(water_speed_magnitudes)
plt.hist(water_speed_magnitudes, bins=50, density=True)
plt.xlabel("velocity magnitude")
plt.ylabel("probability density")
plt.title("velocity distribution for water molecules")

# Close the canvas here; display the completed figure on the next slide.
figure_for_slide = plt.gcf()
plt.close(figure_for_slide)

In [ ]:
display(figure_for_slide)

## Student exercise: Check a proposed bond correlation

Use the saved trajectory to examine the Pt–O versus O–H bond correlation. State the geometric cutoffs and explain how atom tags identify species. Change one selection cutoff and compare the sample or correlation. Describe the evidence and name the extra sampling needed for a mechanistic claim.

Record one prediction, one numerical or graphical result, and your chemical interpretation. Use copies of existing values so the main calculation stays intact.


In [ ]:
# Your code here.
# Use a separate variable for your changed parameter and display the comparison.

### Your answer

Prediction: [write your prediction]

Result: [report a value or refer to your figure]

Chemical explanation: [explain the result]

Assumption or limitation: [state one limitation]


## What have you learned

Let us gather the main ideas from this notebook. You can now:

- Set up ASE dynamics with a pretrained potential and explicit constraints.
- Extract structural measurements from saved trajectory frames.
- Explain how cutoffs and species labels shape a geometric analysis.
- Assess what a short trajectory can tell us about a proposed mechanism.


### Questions for discussion

1. How could a small change to the Pt–O cutoff affect the apparent Pt–O versus O–H correlation?
2. What sampling and thermal checks would you request before treating this trajectory as representative of an experimental interface?
